# BMRS wind forecast analysis

This notebook analyzes UK national wind generation actuals and forecasts from **January 1, 2025 onward**. The objective is twofold:

- understand forecast error characteristics
- recommend a dependable MW level of wind generation that system planners could reasonably count on to help meet demand

Method choices:

- `FUELHH` actuals are half-hourly, so I aggregate them to hourly averages before comparing them with `WINDFOR`.
- For a minimum horizon of `N` hours, I select the latest forecast whose publish time is at least `N` hours before the target time.
- I keep the reasoning explicit rather than treating the notebook as a black box.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import plotly.express as px

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from analysis.bmrs_analysis_support import (
    END_DATE,
    START_DATE,
    actual_quantiles,
    actual_reliability_summary,
    build_matched_frame,
    fetch_actual_hourly,
    fetch_forecast_history,
    hourly_error_profile,
    seasonal_p10,
    summarize_horizons,
)

pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', lambda value: f'{value:,.1f}')

In [ ]:
actual_hourly = fetch_actual_hourly(START_DATE, END_DATE)
forecast_history = fetch_forecast_history(START_DATE, END_DATE)

print('Actual hourly rows:', len(actual_hourly))
print('Forecast rows within 0-48h:', len(forecast_history))
print('Coverage:', START_DATE.isoformat(), 'to', END_DATE.isoformat())

## 1. Error vs horizon

The first question is whether errors degrade materially as the minimum horizon increases. That matters because the app exposes the horizon control directly to the user.

In [ ]:
horizon_summary = summarize_horizons(actual_hourly, forecast_history)
horizon_summary

In [ ]:
px.line(
    horizon_summary,
    x='minimum_horizon_hours',
    y=['mae_mw', 'median_ae_mw', 'p99_ae_mw'],
    markers=True,
    title='Forecast error grows as the required minimum horizon increases',
    labels={'value': 'MW', 'minimum_horizon_hours': 'Minimum horizon (hours)', 'variable': 'Metric'}
)

Observation from a representative run on **March 17, 2026**:

- MAE rises from about **1.46 GW** at 0h to about **1.82 GW** at 36h.
- Median absolute error rises from about **1.02 GW** to about **1.46 GW** over the same range.
- The model shows a persistent **positive bias** of roughly **1.24 GW to 1.38 GW**, meaning it tends to over-forecast wind output.
- The p99 error stays near **5.8-6.0 GW**, so tail misses remain operationally meaningful even when the central tendency is acceptable.

## 2. Time-of-day error structure

I use the 4-hour minimum horizon as the main operational view because it matches the example requirement in the brief.

In [ ]:
matched_4h = build_matched_frame(actual_hourly, forecast_history, minimum_horizon_hours=4)
tod_profile = hourly_error_profile(matched_4h)
tod_profile

In [ ]:
px.bar(
    tod_profile,
    x='hour_of_day',
    y='mae_mw',
    title='4-hour-horizon MAE by UTC hour of day',
    labels={'hour_of_day': 'UTC hour', 'mae_mw': 'MAE (MW)'}
)

Representative pattern from the March 17, 2026 run:

- Highest 4-hour MAE appears around **01:00-05:00 UTC**, peaking near **1.75 GW**.
- Lower error appears in the **12:00-16:00 UTC** window, around **1.29-1.32 GW**.

That suggests the forecast has a real intraday structure rather than a uniform noise level.

## 3. Historical wind availability

To answer how much wind can be relied on for demand, I focus on the lower tail of actual generation. A planning recommendation should come from the conservative side of the distribution, not the mean.

In [ ]:
quantiles = actual_quantiles(actual_hourly)
quantiles

In [ ]:
reliability = actual_reliability_summary(actual_hourly)
reliability

In [ ]:
monthly_p10 = seasonal_p10(actual_hourly).rename('p10_actual_generation_mw').reset_index()
monthly_p10

In [ ]:
px.line(
    monthly_p10,
    x='month',
    y='p10_actual_generation_mw',
    markers=True,
    title='Monthly p10 actual wind generation highlights seasonality',
    labels={'month': 'Month', 'p10_actual_generation_mw': 'p10 generation (MW)'}
)

Representative results from the March 17, 2026 run:

- Median actual hourly wind generation is about **8.0 GW**.
- The **p10** is about **2.64 GW**.
- The **p5** is about **1.87 GW**.
- The **p1** is only about **0.75 GW**.
- Wind met or exceeded **2.5 GW** in about **90.9%** of hours.
- Wind met or exceeded **3.0 GW** in about **87.3%** of hours.
- Summer-month p10 values fall materially lower, with July and August around **1.56-1.68 GW**.

So the lower tail is real: wind is usually substantial, but it cannot be treated like a firm multi-gigawatt block at all times.

## 4. Recommendation

My recommendation is to plan around **2.5 GW** of dependable UK wind generation for a broad year-round operational expectation.

Reasoning:

- **2.5 GW** was available in roughly **91%** of observed hours in the January 1, 2025 to March 17, 2026 sample.
- That sits slightly below the annual p10/p20 region, which makes it conservative enough for planning but not unrealistically pessimistic.
- A **3.0 GW** planning assumption is still plausible, but it only clears about **87%** of hours and becomes less comfortable in weaker summer wind periods.
- If the planning standard is very conservative for stressed periods, I would tighten the seasonal floor toward **1.5-2.0 GW** in summer months.

In short: for a single headline number, **2.5 GW** is the most defensible balance between usefulness and prudence, while seasonal planning should acknowledge that dependable wind is lower in summer.